# Database construction

This notebook constructs the cleaned chat-level dataset used for the subsequent analyses. The raw input consists of individual TSV chat logs containing student–AI interactions.

The main steps are:
1. loading and combining the raw chat logs;
2. reconstructing question–answer pairs and retaining their timestamps;
3. removing unmatched or duplicated question entries;
4. matching AI users to anonymized student identifiers and assessment data;
5. creating chat-level variables, including question and answer length, relevance and homework-copying indicators;
6. aggregating selected usage measures to the student level; and
7. creating a unique identifier for the resulting observations.

The final output is a cleaned chat-level dataframe that serves as the basis for the subsequent profile, descriptive, and regression analyses.

### Load the raw TSV chat logs

In [1]:
import pandas as pd
import glob

folder_path = 'statisztika-szociologia-alapszakosoknak/statisztika-szociologia-alapszakosoknak/'

tsvk = []

for i in range(1, 160):
    filename_pattern = f"{folder_path}{str(i).zfill(3)}-*.tsv"
    file = glob.glob(filename_pattern) # runs the search in the specified folder – the result is a list containing the full paths of the files that meet the criteria
    if len(file) == 1:
        df0 = pd.read_csv(file[0], sep="\t")
        tsvk.append(df0)
    else:
        print(f"Ambiguous or no file for the sequence number: {i}")

In [2]:
print(len(tsvk))
print(type(tsvk[0]))

159
<class 'pandas.core.frame.DataFrame'>


In [3]:
for i, df in enumerate(tsvk):
    null_counts = df.isnull().sum()
    if (null_counts > 0).any():
        print(f"Missing values in chat {i}")
        print(null_counts)
# no output -> no missing values

### Construct the chat-level dataframe

We construct a dataframe in which each row represents a question–answer pair. For each pair, we retain the chat ID, the question sequence number within the chat, the question and answer text, and their timestamps.

In [4]:
import numpy as np
qa_pairs = []

for chat_id, df in enumerate(tsvk, start = 1):
    question_num = 0
    i = 0
    n_rows = df.shape[0]
    while i < n_rows:
        if df.iloc[i]['type'] == 'human':
            question_num += 1
            question_text = df.iloc[i]['content']
            question_time = df.iloc[i]['timestamp']

            # Assume that the next line is the agent's response.
            if i + 1 < n_rows and df.iloc[i+1]['type'] == 'agent':
                answer_text = df.iloc[i+1]['content']
                answer_time = df.iloc[i+1]['timestamp']
                i += 2
            else: # If there is no answer, it will be empty
                answer_text = None
                answer_time = None
                i += 1
            
            # Creating question key-value pairs
            qa_pairs.append({
                'chat_id': chat_id,
                'question_id': question_num,
                'question': question_text,
                'answer': answer_text,
                'q_time': question_time,
                'a_time': answer_time
            })
            
        else: # if the row isn't human
            i += 1

In [5]:
df = pd.DataFrame(qa_pairs)

print(f"Number of question-answer pairs: {len(df)}")
df.info()

total_rows_in_tsvk = sum(chat.shape[0] for chat in tsvk)
print(f"Total rows in raw chat files: {total_rows_in_tsvk}")

Number of question-answer pairs: 1056
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056 entries, 0 to 1055
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   chat_id      1056 non-null   int64 
 1   question_id  1056 non-null   int64 
 2   question     1056 non-null   object
 3   answer       1031 non-null   object
 4   q_time       1056 non-null   object
 5   a_time       1031 non-null   object
dtypes: int64(2), object(4)
memory usage: 49.6+ KB
Total rows in raw chat files: 2112


In [6]:
missing_a = df[df['answer'].isna() | (df['answer'] == '')]

print(f"Number of missing answers: {len(missing_a)}")
print(missing_a[['chat_id', 'question_id', 'question']])

Number of missing answers: 25
     chat_id  question_id                                           question
240       15           12                                          Z érték? 
244       15           16                                  Szórás számítása 
245       15           17                                  Szórás számítása 
246       15           18                                  Szórás számítása 
247       15           19                                  Szórás számítása 
248       15           20                                  Szórás számítása 
249       15           21                                  Szórás számítása 
250       15           22                                  Szórás számítása 
314       25            1                Mit jelent pontosan a standardhiba?
315       25            2                Mit jelent pontosan a standardhiba?
316       25            3                Mit jelent pontosan a standardhiba?
318       25            5                     

There were 25 question entries without an associated AI response. Manual inspection indicated that these cases were caused by duplicated questions in the raw chat data, where the final question had been incorrectly paired with the first response.

### Remove unanswered question entries

In [7]:
df_clean = df.dropna(subset=['answer'])

print(f"Cleaned dataset shape: {df_clean.shape}")

missing_a_clean = df_clean[
    df_clean['answer'].isna() | (df_clean['answer'] == '')
]

print(f"Remaining missing answers: {len(missing_a_clean)}")

Cleaned dataset shape: (1031, 6)
Remaining missing answers: 0


In [8]:
df_clean = df[df['answer'].notna() & (df['answer'] != '')].copy()

Student names were obtained from a separate file and matched to chat IDs. In total, 44 users were successfully matched to students, while 5 users could not be matched. Test scores were also matched to students. Altogether, 993 question–answer pairs remained in the final dataframe. The code and files used for these identification and matching processes are not shared in order to preserve student anonymity. From this point onward, students are referred to using student IDs.

## Create derived variables from the chat-level dataframe

In [9]:
df = pd.read_excel('databases/chatek.xlsx')

In [10]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 993 entries, 0 to 992
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   chat_id                 993 non-null    int64  
 1   question_id             993 non-null    int64  
 2   kérdés                  993 non-null    object 
 3   válasz                  993 non-null    object 
 4   q_time                  993 non-null    object 
 5   a_time                  993 non-null    object 
 6   nev                     993 non-null    object 
 7   ZH1_szazalek            993 non-null    float64
 8   ZH2_szazalek            993 non-null    float64
 9   ZH3_szazalek            993 non-null    float64
 10  ZH4_szazalek            993 non-null    float64
 11  ZH1234                  993 non-null    float64
 12  ZH1234jegy              993 non-null    int64  
 13  ZH1234jegyEGESZJEGYJAV  993 non-null    int64  
 14  ZH1234FELJEGYJAVhoz     993 non-null    fl

In [11]:
df = df.rename(columns={
    'kérdés': 'question',
    'válasz': 'answer',
    'nev': 'id',
    'ZH1_szazalek': 'T1',
    'ZH2_szazalek': 'T2',
    'ZH3_szazalek': 'T3',
    'ZH4_szazalek': 'T4',
    'ZH1234': 'T1234',
})

In [12]:
df = df.drop(columns=[
    'ZH1234jegy',
    'ZH1234jegyEGESZJEGYJAV',
    'ZH1234FELJEGYJAVhoz',
    'ZH1234FELJEGYJAVjegy',
    'MEGAJÁNLOTT JEGY'
])

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 993 entries, 0 to 992
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   chat_id      993 non-null    int64  
 1   question_id  993 non-null    int64  
 2   question     993 non-null    object 
 3   answer       993 non-null    object 
 4   q_time       993 non-null    object 
 5   a_time       993 non-null    object 
 6   id           993 non-null    object 
 7   T1           993 non-null    float64
 8   T2           993 non-null    float64
 9   T3           993 non-null    float64
 10  T4           993 non-null    float64
 11  T1234        993 non-null    float64
dtypes: float64(5), int64(2), object(5)
memory usage: 93.2+ KB


### Calculate question and answer length

In [14]:
df['question_length'] = df['question'].apply(len)
df['answer_length'] = df['answer'].apply(len)

print(df['question_length'].describe())
print(df['answer_length'].describe())

count     993.000000
mean      140.294058
std       209.817918
min         4.000000
25%        32.000000
50%        58.000000
75%       139.000000
max      1722.000000
Name: question_length, dtype: float64
count     993.000000
mean     1292.305136
std       647.986158
min        23.000000
25%       866.000000
50%      1307.000000
75%      1719.000000
max      3676.000000
Name: answer_length, dtype: float64


Very long questions were typically associated with copied textbook exercises and were therefore relevant for the subsequent homework-copying classification.

Relevance labels and duplicate indicators were manually reviewed and revised. The updated dataset is loaded below for the subsequent processing steps.

In [15]:
df = pd.read_excel('databases/chatek_id_1.xlsx')

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 993 entries, 0 to 992
Data columns (total 25 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   chat_id                              993 non-null    int64  
 1   question_id                          993 non-null    int64  
 2   kérdés                               993 non-null    object 
 3   válasz                               993 non-null    object 
 4   q_time                               993 non-null    object 
 5   a_time                               993 non-null    object 
 6   id                                   993 non-null    object 
 7   ZH1_szazalek                         993 non-null    float64
 8   ZH2_szazalek                         993 non-null    float64
 9   ZH3_szazalek                         993 non-null    float64
 10  ZH4_szazalek                         993 non-null    float64
 11  ZH1234                          

In [17]:
df = df[df['duplum'].isna()].copy()
df = df.drop(columns=['duplum'])

After filtering out further duplicates 990 question-answer pairs remained. 

### Calculate the number of questions and chats per student

In [18]:
question_counts = df.groupby('id').size().reset_index(name='question_count')

df = df.merge(question_counts, on='id', how='left')

chat_counts = df.groupby('id')['chat_id'].nunique().reset_index(name='chat_count')

df = df.merge(chat_counts, on='id', how='left')

In [19]:
df = df.rename(columns={
    'kérdés': 'question',
    'válasz': 'answer',
    'ZH1_szazalek': 'T1',
    'ZH2_szazalek': 'T2',
    'ZH3_szazalek': 'T3',
    'ZH4_szazalek': 'T4',
    'ZH1234': 'T1234',
    'kérdéshossz': 'question_length',
    'válaszhossz': 'answer_length',
    'kérdésszám': 'question_count',
    'chatszám': 'chat_count',
    'szakmai relevancia (1 igen / 0 nem)': 'relevance'
})
df = df.drop(columns=[
    'ZH1234jegy',
    'ZH1234jegyEGESZJEGYJAV',
    'ZH1234FELJEGYJAVhoz',
    'ZH1234FELJEGYJAVjegy',
    'MEGAJÁNLOTT JEGY',
    'kérdés_nszavak',
    'válasz_nszavak'
])

In [ ]:
print(df['relevance'].value_counts())

### Calculate average question and answer length at the student level

In [32]:
avg_lengths = df.groupby('id').agg(
    avg_question_length=('question_length', 'mean'),
    avg_answer_length=('answer_length', 'mean')
).reset_index()

df = df.merge(avg_lengths, on='id', how='left')

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   chat_id              990 non-null    int64  
 1   question_id          990 non-null    int64  
 2   question             990 non-null    object 
 3   answer               990 non-null    object 
 4   q_time               990 non-null    object 
 5   a_time               990 non-null    object 
 6   id                   990 non-null    object 
 7   T1                   990 non-null    float64
 8   T2                   990 non-null    float64
 9   T3                   990 non-null    float64
 10  T4                   990 non-null    float64
 11  T1234                990 non-null    float64
 12  question_length      990 non-null    int64  
 13  answer_length        990 non-null    int64  
 14  question_count       990 non-null    int64  
 15  chat_count           990 non-null    int

### Create a unique row identifier

In [34]:
df['row_id'] = range(1, len(df) + 1)

In [35]:
print(df.shape)
print(df['row_id'].min(), df['row_id'].max())
print(df['row_id'].nunique())

(990, 20)
1 990
990


In [36]:
df.to_excel('databases/chat_database_clean.xlsx', index=False)